<a href="https://colab.research.google.com/github/Peiprjs/MAI3002-Framingham_Heart_Study/blob/main/Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing
## Imports (Packages and Data)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from colorama import Fore, Back, Style

from scipy import stats
from scipy.stats import levene

import numpy as np
from numpy.ma.core import indices

from sklearn import tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Change False to True during development, they are used to ignore warnings when in "prod".
import warnings
if True: warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/LUCE-Blockchain/Databases-for-teaching/refs/heads/main/Framingham%20Dataset.csv')

### Function imports

#### Graphs
##### Correlation plot

In [ ]:
def corr_plot(input_df):
    corr_df = input_df.corr()
    corr_mat = corr_df[corr_df.columns[::-1]].stack().reset_index(name="correlation")

    sns.set_style("whitegrid")
    g = sns.relplot(
        data=corr_mat,
        x="level_0", y="level_1", hue="correlation", size="correlation",
        palette="vlag", edgecolor=".7",
        height=10, sizes=(50, 250), hue_norm=(-0.5, 1),size_norm=(-0.5, 1),
    )

    g.set(xlabel="", ylabel="", aspect="equal")
    g.despine(left=True, bottom=True)
    g.ax.margins(.02)
    for label in g.ax.get_xticklabels():
        label.set_rotation(90)

##### Distplots


In [ ]:
def distplots(df):
    numeric_df = df.select_dtypes(include=['number'])
    num_features = len(numeric_df.columns)
    cols = int(np.ceil(np.sqrt(num_features)))
    rows = int(np.ceil(num_features / cols))

    # A figure with subplots looks much nicer
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
    axes = axes.flatten()

    for i, column in enumerate(numeric_df.columns):
        # Technically not needed but might as well
        numeric_df_nona = numeric_df[column].dropna()

        axes[i].hist(numeric_df_nona, bins=30, alpha=0.7, edgecolor='black')

        if len(numeric_df_nona) > 1:
            density = stats.gaussian_kde(numeric_df_nona)
            xs = np.linspace(numeric_df_nona.min(), numeric_df_nona.max(), 200)
            axes[i].plot(xs, density(xs) * len(numeric_df_nona) * (numeric_df_nona.max() - numeric_df_nona.min()) / 30,
                         'r-', linewidth=2)

        axes[i].set_xlabel(column)
        axes[i].set_ylabel('Number of Patients')
        axes[i].set_title(f'Distribution of {column}')
        axes[i].grid(axis='y', alpha=0.3)

    # Remove any empty subplots if they exist
    for j in range(num_features, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

#### Imputation
##### Drop high missing columns

In [ ]:
def drop_high_missing_cols(df, threshold=0.50):
    """
    1. Drops columns with a high percentage of missing values.
    """
    print(f"--- Running Step 1: Dropping columns > {threshold:.0%} missing ---")
    # Calculate missing percentage on the dataframe
    missing_pct = df.isnull().sum() / len(df)

    # Identify columns to drop based on the threshold
    cols_to_drop = missing_pct[missing_pct >= threshold].index.tolist()

    if cols_to_drop:
        print(f"   -> Dropping columns: {', '.join(cols_to_drop)}")
        # Drop columns from the dataframe
        df_dropped = df.drop(columns=cols_to_drop)
    else:
        print("   -> No columns exceeded the missing value threshold.")
        df_dropped = df.copy()

    return df_dropped


##### Dataframe reconstruction

In [ ]:
def _reconstruct_dataframe(encoded_df, original_df, num_cols, cat_cols, enc):
    """
    Internal helper function to revert a one-hot encoded DataFrame
    back to its original shape.
    """
    # Isolate the imputed numerical data
    # Ensure we only select num_cols that are still in the encoded_df
    present_num_cols = [col for col in num_cols if col in encoded_df.columns]
    imputed_numerical = encoded_df[present_num_cols]

    # Isolate the encoded columns to be inverse-transformed
    encoded_cols_names = enc.get_feature_names_out(cat_cols)

    # Ensure all expected encoded columns are present, fill with 0 if not
    for col in encoded_cols_names:
        if col not in encoded_df.columns:
            encoded_df[col] = 0

    imputed_encoded = encoded_df[encoded_cols_names]

    # Perform the inverse transform
    imputed_categorical_array = enc.inverse_transform(imputed_encoded)

    # Convert the result back to a DataFrame
    imputed_categorical = pd.DataFrame(imputed_categorical_array,
                                       columns=cat_cols,
                                       index=encoded_df.index)

    # Combine numerical and reverted categorical data
    reconstructed_df = pd.concat([imputed_numerical, imputed_categorical], axis=1)

    # Enforce the original column order
    # Use .columns.intersection() to avoid errors if columns were dropped
    original_cols_present = original_df.columns.intersection(reconstructed_df.columns)
    final_df = reconstructed_df.reindex(columns=original_cols_present)

    return final_df

##### KNN imputation

In [ ]:
def knn_impute(df, min_thresh=0.05, max_thresh=0.50, n_neighbors=3):
    """
    2. Uses KNN Imputation for columns with moderate missing values (5%-50%).

    What the function does:
    - One-hot encodes categorical data.
    - Trains KNN models (Regressor or Classifier) on the data
      to predict its own missing values.
    - Reconstructs the dataframe back to its original format.
    """
    print(f"\n--- Running Step 2: KNN Imputation ({min_thresh:.0%} - {max_thresh:.0%} missing) ---")
    # Create a copy to avoid modifying the original dataframe
    df_imputed = df.copy()

    # --- 1. One-Hot Encoding ---

    # Identify categorical/numerical columns
    categorical_cols = [col for col in df_imputed.columns
                        if df_imputed[col].dtype == 'object']
    numerical_cols = [col for col in df_imputed.columns
                      if df_imputed[col].dtype != 'object']

    # Initialize encoder
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    # Handle case with no categorical columns
    if not categorical_cols:
        print("   -> No categorical columns found. Skipping encoding.")
        df_encoded = df_imputed.copy()
        encoder = None  # Flag that no encoder was used
    else:
        print(f"   -> Fitting OneHotEncoder on {len(categorical_cols)} columns...")
        # Fit and transform the data
        encoded_cols_df = pd.DataFrame(encoder.fit_transform(df_imputed[categorical_cols]),
                                       columns=encoder.get_feature_names_out(categorical_cols),
                                       index=df_imputed.index)

        # Create the new, fully encoded dataframe
        df_encoded = df_imputed.drop(columns=categorical_cols).join(encoded_cols_df)
        print(f"   -> Data encoded. New shape: {df_encoded.shape}")

    # --- 2. KNN Imputation on Data ---

    # Calculate means *once* to be used for filling features (not targets)
    feature_fill_values = df_encoded.mean()

    for col in df_encoded.columns:
        missing_pct = df_encoded[col].isnull().mean()

        # Apply KNN to columns with moderate missingness
        if min_thresh <= missing_pct <= max_thresh:
            print(f"   -> KNN Imputing '{col}' (Missing: {missing_pct:.2%})")

            other_cols = [c for c in df_encoded.columns if c != col]

            # Split data for the imputer model
            train_rows = df_encoded[col].notnull()
            predict_rows = df_encoded[col].isnull()

            # If no rows to predict, skip
            if not predict_rows.any():
                print(f"      -> Skipping '{col}', no rows to predict.")
                continue

            # Fill NaNs in *features* with the mean for model stability
            X_train = df_encoded.loc[train_rows, other_cols].fillna(feature_fill_values)
            y_train = df_encoded.loc[train_rows, col]
            X_predict = df_encoded.loc[predict_rows, other_cols].fillna(feature_fill_values)

            # Select and train the appropriate KNN model
            is_binary = y_train.nunique() <= 2
            model = KNeighborsClassifier(n_neighbors=n_neighbors) if is_binary else KNeighborsRegressor(
                n_neighbors=n_neighbors)

            model.fit(X_train, y_train)
            predicted_values = model.predict(X_predict)

            # Assign the predicted values back to the encoded dataframe
            df_encoded.loc[predict_rows, col] = predicted_values
            print(f"      -> Successfully imputed {len(predicted_values)} values.")

    # --- 3. Reconstruct DataFrame ---

    # If no encoding was done, just return the dataframe
    if encoder is None:
        print("   -> Skipping reconstruction (no categorical columns).")
        return df_encoded

    print("   -> Reconstructing dataframe...")
    # Reconstruct the dataframe
    df_reconstructed = _reconstruct_dataframe(
        df_encoded,
        df_imputed,  # Pass the original copy for column order
        numerical_cols,
        categorical_cols,
        encoder
    )
    df_reconstructed.sort_index(inplace=True)

    return df_reconstructed

##### Normal imputation

In [ ]:
def impute_simple_central(df, max_thresh=0.05):
    """
    3. Uses simple imputation (median/mode) for columns with < 5% missing values.

    This function learns the imputation value (median for numeric, mode for
    categorical) from the dataframe and applies it to fill its own NaNs.
    """
    print(f"\n--- Running Step 3: Simple Imputation (< {max_thresh:.0%} missing) ---")
    df_imputed = df.copy()

    for col in df_imputed.columns:
        # Calculate the percentage of missing values
        missing_pct = df_imputed[col].isnull().mean()

        # Check if the column fits the < 5% criteria
        if 0 < missing_pct < max_thresh:
            print(f"   -> Found '{col}' with {missing_pct:.2%} missing values. Imputing...")

            # Distinguish between numerical and categorical data
            if pd.api.types.is_numeric_dtype(df_imputed[col]):
                # For numerical columns, use the median
                fill_value = df_imputed[col].median()
                df_imputed[col].fillna(fill_value, inplace=True)
                print(f"      -> Filled with median: {fill_value}")
            else:
                # For categorical columns, use the mode
                fill_value = df_imputed[col].mode()[0]
                df_imputed[col].fillna(fill_value, inplace=True)
                print(f"      -> Filled with mode: '{fill_value}'")

    print("   -> Simple imputation complete.")
    return df_imputed

#### Data cleanup

In [ ]:
def skewness_check(df):
    for column in df:
        skewness = stats.skew(df[column])
        if skewness > 1:
            print(Fore.RED + f"{column} is strongly right skewed (skew: {skewness:.3f})")
        elif skewness > 0.5:
            print(Fore.RED + f"{column} is moderately right skewed (skew: {skewness:.3f})")
        elif stats.skew(data_imputed[column]) < -1:
            print(Fore.MAGENTA + f"{column} is strongly left skewed (skew: {skewness:.3f})")
        elif stats.skew(data_imputed[column]) < -0.5:
            print(Fore.MAGENTA + f"{column} is moderately left skewed (skew: {skewness:.3f})")
        else:
            print(Fore.GREEN + f"{column} is symmetric (skew: {skewness:.3f})")
print(Style.RESET_ALL)

## Research questions
- Is there a statistically significant difference in mean total cholesterol levels between current smokers and non-smokers?
- Is there a statistically significant difference in mean blood pressure levels between current smokers and non-smokers?
- Among individuals who have developed cardiovascular disease, is there a statistically significant difference in mean age between current smokers and non-smokers?
- Among individuals who have suffered any kind of cardiovascular disease, is there a statistically significant difference in mean age between diabetic and non-diabetic people?

## Data exploration and cleanup

In [ ]:
numeric_df = data.select_dtypes(include=['number'])
time_cols = data.columns[data.columns.str.startswith('TIME')]

In [ ]:
data.shape

### Statistics

In [ ]:
data.describe()

### Graphs

In [ ]:
distplots(data)

In [ ]:
corr_plot(data.dropna())

In [ ]:
binary_cols = [col for col in data.columns if
               data[col].dropna().isin([0, 1, 2, 3, 4]).all() and len(data[col].dropna().unique()) <= 4]

num_binary = len(binary_cols)
cols = int(np.ceil(np.sqrt(num_binary)))
rows = int(np.ceil(num_binary / cols))

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten() if num_binary > 1 else [axes]

for i, col in enumerate(binary_cols):
    value_counts = data[col].value_counts()
    axes[i].pie(value_counts, labels=value_counts.index, autopct='%1.1f%%', startangle=90)
    axes[i].set_title(f'{col}')

for j in range(num_binary, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
distplots(numeric_df.drop(binary_cols, axis=1))

⚠️ Will take a long time to run

In [ ]:
if False: sns.pairplot(data, hue='ANGINA')

### Detecting outliers in the numerical columns

In [ ]:
numerical_cols = data.select_dtypes(include=np.number).columns
outlier_indices = {}

for col in numerical_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    indices = data[(data[col] < lower_bound) | (data[col] > upper_bound)].index
    if len(indices) > 0:
            outlier_indices[col] = indices.tolist()
            #print(f'{len(indices)} outliers found in {col} column')
    else:
        pass
        #print(f"No outlier data found for column {col}")

outlier_values = []
outliers_count = 0
for col, rows in outlier_indices.items():
    outlier_values.append([col, len(rows), (len(rows) / len(data[col]))])
    outliers_count = outliers_count + 1 if len(rows) > 0 else outliers_count

if outliers_count > 0:
    data_outliers = pd.DataFrame(outlier_values)
    data_outliers.columns = ['Variable', 'Outliers', 'Percentage Outliers']
    s = data_outliers.sort_values(by=['Percentage Outliers'], ascending=False).style.bar(subset=['Percentage Outliers'], color='#d65f5f')
    display(s)
else:
    print('No outlier values found in the dataset.')

### Data imputation

In [ ]:
missing  = 0
misVariables = []
CheckNull = data.isnull().sum()
for var in range(0, len(CheckNull)):
    misVariables.append([data.columns[var], CheckNull[var], round(CheckNull[var]/len(data),3)])
    missing = missing + 1

if missing == 0:
    print('Dataset is complete with no blanks.')
else:
    data_misVariables = pd.DataFrame.from_records(misVariables)
    data_misVariables.columns = ['Variable', 'Missing', 'Percentage missing']
    s = data_misVariables.sort_values(by=['Percentage missing'], ascending=False).style.bar(subset=['Percentage missing'], color='#d65f5f')
    display(s)

Dropping columns with more than 50% missing data

In [ ]:
data_dropped = drop_high_missing_cols(data, threshold=0.50)

Using KNN Classifier or Regressor (based on data category) on columns between 2% and 50%

In [ ]:
data_knn_predicted= knn_impute(data_dropped, min_thresh=0.02, max_thresh=0.50, n_neighbors=5)

Using measures of central tendency on columns where <2% of data is missing

In [ ]:
from imputation_functions import impute_simple_central

data_imputed = impute_simple_central(data_knn_predicted)

Code for showing if the imputation successful was

In [ ]:
missing  = 0
misVariables = []
CheckNull = data_imputed.isnull().sum()
for var in range(0, len(CheckNull)):
    misVariables.append([data_imputed.columns[var], CheckNull[var], round(CheckNull[var]/len(data),3)])
    missing = missing + 1

if missing == 0:
    print('Dataset is complete with no blanks.')
else:
    data_misVariables = pd.DataFrame.from_records(misVariables)
    data_misVariables.columns = ['Variable', 'Missing', 'Percentage missing']
    s = data_misVariables.sort_values(by=['Percentage missing'], ascending=False).style.bar(subset=['Percentage missing'], color='#d65f5f')
    display(s)

In [ ]:
corr_plot(data.drop(columns=['HDLC','LDLC']))
corr_plot(data_imputed)

In [ ]:
for column in data_imputed:
    defined_variable = column
    original = data[defined_variable]
    imputed = data_imputed[defined_variable]

    fig, axes = plt.subplots(1, 2, figsize=(20, 4))

    axes[0].hist(original.dropna(), bins=30, alpha=0.7, color='#5D3A9B', edgecolor='black')
    if len(original.dropna()) > 1:
        from scipy import stats

        density = stats.gaussian_kde(original.dropna())
        xs = np.linspace(original.min(), original.max(), 200)
        axes[0].plot(xs, density(xs) * len(original.dropna()) * (original.max() - original.min()) / 30, 'r-',
                     linewidth=2)
    axes[0].set_xlabel(defined_variable)
    axes[0].set_ylabel('Density')
    axes[0].set_title('Original')
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].hist(imputed, bins=30, alpha=0.7, color='#E66100', edgecolor='black')
    if len(imputed) > 1:
        density = stats.gaussian_kde(imputed)
        xs = np.linspace(imputed.min(), imputed.max(), 200)
        axes[1].plot(xs, density(xs) * len(imputed) * (imputed.max() - imputed.min()) / 30, 'r-', linewidth=2)
    axes[1].set_xlabel(defined_variable)
    axes[1].set_ylabel('Density')
    axes[1].set_title('Imputed')
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Distribution Comparison of {defined_variable}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Data skewness

In [ ]:
# TODO: check that it works properly

In [ ]:
data_imputed_unskewed = data_imputed.copy().drop(binary_cols, axis=1).drop(time_cols, axis=1)

for column in data_imputed_unskewed:
    skewness = stats.skew(data_imputed_unskewed[column])
    max_val = data_imputed[column].max()
    if skewness > 1:
        print(Fore.RED + f"{column} is strongly right skewed (skew: {skewness:.3f})")
        data_imputed_unskewed[column] = np.log1p(data_imputed_unskewed[column])
    elif skewness > 0.5:
        print(Fore.RED + f"{column} is moderately right skewed (skew: {skewness:.3f})")
        data_imputed_unskewed[column] = np.log1p(data_imputed_unskewed[column])
    elif stats.skew(data_imputed[column]) < -1:
        print(Fore.MAGENTA + f"{column} is strongly left skewed (skew: {skewness:.3f})")
        data_imputed_unskewed[column] = np.power(max_val + 1 - data_imputed_unskewed[column], 2)
    elif stats.skew(data_imputed[column]) < -0.5:
        print(Fore.MAGENTA + f"{column} is moderately left skewed (skew: {skewness:.3f})")
        data_imputed_unskewed[column] = np.power(max_val + 1 - data_imputed_unskewed[column], 2)
    else:
        print(Fore.GREEN + f"{column} is symmetric (skew: {skewness:.3f})")

In [ ]:
skewness_check(data_imputed_unskewed)

In [ ]:
distplots(data_imputed_unskewed)

In [ ]:
distplots(data.drop(binary_cols, axis=1).drop(time_cols, axis=1).drop(columns=['HDLC','LDLC']))

In [ ]:
data_imputed_unskewed = pd.concat([data_imputed_unskewed, data_imputed[time_cols], data_imputed[binary_cols]], axis=1)

## Research question exploration

### Exploring how the cholesterol differs between Smokers and Non-smokers

In [ ]:
mean_cholesterol = data_imputed.groupby('CURSMOKE')['TOTCHOL'].describe()
print("Mean Total Cholesterol:")
print(mean_cholesterol)

nonsmokers_chol = data_imputed[data_imputed['CURSMOKE'] == 0]['TOTCHOL']
smokers_chol = data_imputed[data_imputed['CURSMOKE'] == 1]['TOTCHOL']
_, levene_p = stats.levene(nonsmokers_chol, smokers_chol)

t_stat, p_value = stats.ttest_ind(smokers_chol, nonsmokers_chol,
                                  equal_var=False if levene_p < 0.05 else True)
print("\nIndependent T-Test Results:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: The difference in mean total cholesterol is statistically significant.\n")
else:
    print("Result: There is no statistically significant difference in mean total cholesterol.\n")

In [ ]:
plt.figure(figsize=(8, 6))
ax = sns.boxplot(x='CURSMOKE', y='TOTCHOL', data=data_imputed)


# Improve the plot labels
plt.title('Total Cholesterol by Smoking Status', fontsize=16)
plt.ylabel('Total Cholesterol', fontsize=12)
plt.xlabel('Current Smoker', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Non-Smoker', 'Smoker'])

plt.text(0.5, 0.94, "p < 0.001" if p_value < 0.001 else round(p_value, 3),
         horizontalalignment='center',
         verticalalignment='top',
         transform=ax.transAxes, # Use axes-relative coordinates
         fontsize=12,
         bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.2'))

plt.show()
plt.close()

### Exploring how the blood pressure differs between Smokers and Non-smokers


In [ ]:
mean_sys = data_imputed.groupby('CURSMOKE')['SYSBP'].describe()
mean_dias = data_imputed.groupby('CURSMOKE')['DIABP'].describe()

nonsmokers_sys = data_imputed[data_imputed['CURSMOKE'] == 0]['SYSBP']
smokers_sys = data_imputed[data_imputed['CURSMOKE'] == 1]['SYSBP']
nonsmokers_dias = data_imputed[data_imputed['CURSMOKE'] == 0]['DIABP']
smokers_dias = data_imputed[data_imputed['CURSMOKE'] == 1]['DIABP']

# Systolic blood pressure
t_stat, p_value = stats.ttest_ind(nonsmokers_sys, smokers_sys,
                                  equal_var=False if levene_p < 0.05 else True)
print("\nIndependent T-Test Results:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: The difference in mean systolic blood pressure is statistically significant.\n")
else:
    print("Result: There is no statistically significant difference in mean systolic blood pressure.\n")

# Diastolic blood pressure
t_stat, p_value = stats.ttest_ind(nonsmokers_dias, smokers_dias,
                                  equal_var=False if levene_p < 0.05 else True)
print("\nIndependent T-Test Results:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: The difference in mean diastolic blood pressure is statistically significant.\n")
else:
    print("Result: There is no statistically significant difference in mean diastolic blood pressure.\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='CURSMOKE', y='SYSBP', data=data_imputed, ax=axes[0])
axes[0].set_title('Systolic blood pressure by Smoking Status', fontsize=16)
axes[0].set_ylabel('Systolic blood pressure (mmHg)', fontsize=12)
axes[0].set_xlabel('Current Smoker', fontsize=12)
axes[0].set_xticks(ticks=[0, 1])
axes[0].set_xticklabels(['Non-Smoker', 'Smoker'])

axes[0].text(0.5, 0.94, "p < 0.001" if p_value < 0.001 else round(p_value, 3),
             horizontalalignment='center',
             verticalalignment='top',
             transform=axes[0].transAxes,
             fontsize=12,
             bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.2'))

sns.boxplot(x='CURSMOKE', y='DIABP', data=data_imputed, ax=axes[1])
axes[1].set_title('Diastolic blood pressure by Smoking Status', fontsize=16)
axes[1].set_ylabel('Diastolic blood pressure (mmHg)', fontsize=12)
axes[1].set_xlabel('Current Smoker', fontsize=12)
axes[1].set_xticks(ticks=[0, 1])
axes[1].set_xticklabels(['Non-Smoker', 'Smoker'])

axes[1].text(0.5, 0.94, "p < 0.001" if p_value < 0.001 else round(p_value, 3),
             horizontalalignment='center',
             verticalalignment='top',
             transform=axes[1].transAxes,
             fontsize=12,
             bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.2'))

plt.tight_layout()
plt.show()
plt.close()

### Exploring how age differs for people developed Cardiovascular Disease based on their smoking status?

In [ ]:
patients_with_cvd = data_imputed[data_imputed['CVD'] == 1].copy()
if patients_with_cvd.empty:
    raise ValueError("No individuals with CVD found in the dataset.")

age_cvd_per_smoker_group = patients_with_cvd.groupby('CURSMOKE')['AGE'].describe()
print("Age Statistics for Patients with CVD:")
print(age_cvd_per_smoker_group)
print("\n")

smokers_age_with_cvd = patients_with_cvd[patients_with_cvd['CURSMOKE'] == 1]['AGE']
nonsmokers_age_without_cvd = patients_with_cvd[patients_with_cvd['CURSMOKE'] == 0]['AGE']

if smokers_age_with_cvd.empty or smokers_age_with_cvd.empty:
    raise ValueError("Not enough data for one or both groups to run a t-test.")

_, levene_p = stats.levene(smokers_age_with_cvd, nonsmokers_age_without_cvd)

t_stat, p_value = stats.ttest_ind(smokers_age_with_cvd, nonsmokers_age_without_cvd,
                          equal_var=False if levene_p < 0.05 else True)
print("\nIndependent T-Test Results:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}\n")
if p_value < 0.05:
    print("Result: Among CVD patients, there is a significant age difference between smokers and non-smokers.\n")
else:
    print("Result: Among CVD patients, there is no significant age difference between smokers and non-smokers.\n")

plt.figure(figsize=(8, 6))

ax = sns.boxplot(x='CURSMOKE', y='AGE', data=patients_with_cvd)

# To see individual data points
sns.stripplot(x='CURSMOKE', y='AGE', data=patients_with_cvd, color=".25", alpha=0.3)

plt.title('Age of Patients with Cardiovascular Disease by Smoking Status', fontsize=16)
plt.ylabel('Age', fontsize=12)
plt.xlabel('Current Smoker', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Non-Smoker', 'Smoker'])

plt.text(0.5, 0.91, "p < 0.001" if p_value < 0.001 else round(p_value, 3),
         horizontalalignment='center',
         verticalalignment='top',
         transform=ax.transAxes, # Use axes-relative coordinates
         fontsize=12,
         bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.2'))

plt.show()
plt.close()


# Machine learning

## Train, test split

In [ ]:
X, y = data_imputed_unskewed.drop(columns='ANYCHD'), data_imputed_unskewed['ANYCHD']

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.20, stratify=y, random_state=2025)


## Logistic Regression

### Baseline model

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
lr_baseline = LogisticRegression(max_iter=1000, random_state=2025)
lr_baseline.fit(train_X, train_y)
y_pred_base = lr_baseline.predict(test_X)

print(f"Baseline Accuracy: {f1_score(test_y, y_pred_base):.4f}")

Baseline Accuracy: 0.9309


## Random forest

In [ ]:
X, y = data_imputed_unskewed[['SEX', 'TOTCHOL', 'AGE', 'SYSBP', 'DIABP', 'CURSMOKE', 'CIGPDAY', 'BMI', 'DIABETES', 'BPMEDS', 'GLUCOSE', 'PREVCHD']], data_imputed_unskewed['ANYCHD']

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.20, stratify=y, random_state=2025)

clf = tree.DecisionTreeClassifier(criterion='entropy', random_state=2023, class_weight='balanced')
clf = clf.fit(train_X, train_y)
prediction = clf.predict(test_X)

print('Accuracy: ', accuracy_score(y_true=test_y, y_pred=prediction))

cm = confusion_matrix(y_true=test_y, y_pred=prediction, normalize='true')

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot();


In [ ]:
fig_tree, ax = plt.subplots()

tree.plot_tree(clf, ax=ax, impurity = False, feature_names= X.columns, class_names= ("Died", "Survived") ,proportion = True, rounded = True, precision = 2, filled = True);

plt.savefig('TreePlot.pdf')